# Ausgrid experiment 3 — daily load-total calibration

**Hypothesis:** Training-only calibration of GC+CL daily energy will reduce the daily-load Wasserstein distance of `diffusion_v2_post` without losing its peak, solar, or utility gains. The comparison changes only this one postprocessing step; the paired daily profiles, diffusion checkpoints, night mask, household split, and evaluator stay the same.

| Arm | Seeds | GC+CL daily-total mapping | GG |
| --- | --- | --- | --- |
| `diffusion_v2_post` | 1, 2, 3 | none | existing night mask |
| `diffusion_v2_loadcal_post` | 1, 2, 3 | fitted on the 30,000 **training-household** contexts per seed | copied exactly |

Put this notebook in `notebooks/` beside `Ausgrid_Diffusion_Colab.ipynb`. Run it in VS Code with a regular Python kernel, including a Colab kernel if convenient. It needs `numpy`, `pandas`, `scipy`, `scikit-learn`, and `matplotlib`; **no GPU or model retraining is needed**. It reads the existing `data/prepared/prepared.npz` and `outputs/samples/diffusion_v2_post_seed{1,2,3}_{train,test,privacy}.npz`. Run the diffusion-v2 notebook through its sampling checkpoint first. No Drive mounting or zip files.

The held-out test households have already been viewed in previous experiments. Treat this follow-up as **exploratory**, and report all three seeds and adverse changes rather than tuning on the held-out scores. This calibrates a marginal distribution; it cannot by itself prove that conditional relationships or privacy are preserved.


## Checkpoint 0 — project and input inventory

In [ ]:
from pathlib import Path
import sys, json, hashlib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import wasserstein_distance

cwd = Path.cwd().resolve()
PROJECT_ROOT = next((p for p in (cwd, cwd.parent, cwd/'ausgrid-synthetic')
                     if (p/'data/prepared/prepared.npz').is_file()
                     and (p/'src/ausgrid_synth/evaluate.py').is_file()), None)
assert PROJECT_ROOT is not None, 'Open the notebook from the project notebooks/ folder.'
sys.path.insert(0, str(PROJECT_ROOT/'src'))
from ausgrid_synth.data import load_prepared
from ausgrid_synth import evaluate as existing_evaluation

PREPARED = PROJECT_ROOT/'data/prepared/prepared.npz'
SAMPLES = PROJECT_ROOT/'outputs/samples'
REPORTS = PROJECT_ROOT/'outputs/reports'
REPORTS.mkdir(parents=True, exist_ok=True)
DATA = load_prepared(PREPARED)
assert DATA['x'].shape[1:] == (2,48)
assert sorted(np.unique(DATA['split']).tolist()) == [0,1,2]
BASE_ARM = 'diffusion_v2_post'
CAL_ARM = 'diffusion_v2_loadcal_post'
SEEDS = (1,2,3)
QUANTILES = np.linspace(.001,.999,999,dtype='float64')
METHOD = 'daily_gccl_training_ecdf_ratio_tail_v1'

def sha256_file(path):
    digest=hashlib.sha256()
    with path.open('rb') as stream:
        for block in iter(lambda: stream.read(4*1024*1024), b''):
            digest.update(block)
    return digest.hexdigest()

DATA_SHA256 = sha256_file(PREPARED)
expected = [SAMPLES/f'{BASE_ARM}_seed{s}_{c}.npz'
            for s in SEEDS for c in ('train','test','privacy')]
missing = [str(path) for path in expected if not path.is_file()]
assert not missing, 'Run the diffusion-v2 sampling checkpoint first. Missing:\n'+'\n'.join(missing)
print('Project:',PROJECT_ROOT)
print('Paired days:',len(DATA['x']),'| source sample files:',len(expected))
print('Separate output arm:',CAL_ARM)


## One-factor mapping

For each seed, use only its generated **training** daily GC+CL totals and the real totals for the same training context indices. Match their empirical quantiles from 0.1% to 99.9%. Outside that range, extrapolate by the endpoint multiplicative ratio so the extreme days are visible instead of silently clipping them. Rescale all 48 GC+CL intervals of each generated day by `mapped total / original total`. GG stays byte-for-byte identical in memory, and zero load stays zero. This preserves each nonzero load day's *relative* shape, but can change peak magnitudes and GC–GG dependence; those changes are measured below.

The mapping never reads real validation or test profiles during fitting. Its training-set Wasserstein distance is a fit diagnostic, **not** an independent result.


In [ ]:
def context_indices(context):
    rng=np.random.default_rng(2026)
    if context=='train':
        available=np.flatnonzero(DATA['split']==0)
        return rng.choice(available,size=min(30000,len(available)),replace=False)
    if context=='test': return np.flatnonzero(DATA['split']==2)
    if context=='privacy':
        available=np.flatnonzero(np.isin(DATA['split'],[0,2]))
        return rng.choice(available,size=min(8000,len(available)),replace=False)
    raise ValueError(context)

def load_source(seed,context):
    path=SAMPLES/f'{BASE_ARM}_seed{seed}_{context}.npz'
    with np.load(path,allow_pickle=False) as archive:
        x,idx=archive['x'],archive['idx']
    assert x.shape==(len(idx),2,48) and np.array_equal(idx,context_indices(context)), (seed,context)
    assert np.isfinite(x).all() and np.min(x)>=0, (seed,context)
    assert np.all(x[:,1,:][DATA['night'][idx]]==0), 'Expected the night-masked source arm.'
    return x,idx

def fit_mapping(seed):
    source,idx=load_source(seed,'train')
    generated=source[:,0,:].sum(axis=1,dtype=np.float64)
    observed=DATA['x'][idx,0,:].sum(axis=1,dtype=np.float64)
    assert generated.max()>0 and observed.max()>0
    src_q=np.quantile(generated,QUANTILES)
    dst_q=np.quantile(observed,QUANTILES)
    unique,first=np.unique(src_q,return_index=True)
    positive=unique>0
    unique,target=unique[positive],dst_q[first][positive]
    assert len(unique)>1 and np.all(np.diff(target)>=0)
    return {'method':METHOD,'seed':seed,'data_sha256':DATA_SHA256,
            'source_arm':BASE_ARM,'source_train_sha256':sha256_file(SAMPLES/f'{BASE_ARM}_seed{seed}_train.npz'),
            'src_q':unique.tolist(),'dst_q':target.tolist(),
            'source_zero_days':int((generated==0).sum()),
            'real_zero_days':int((observed==0).sum()),
            'training_days':len(idx)}

def mapped_total(total,mapping):
    total=np.asarray(total,dtype=np.float64)
    src=np.asarray(mapping['src_q'],dtype=np.float64)
    dst=np.asarray(mapping['dst_q'],dtype=np.float64)
    mapped=np.interp(total,src,dst)
    mapped=np.where(total<src[0],total*(dst[0]/src[0]),mapped)
    mapped=np.where(total>src[-1],total*(dst[-1]/src[-1]),mapped)
    assert np.isfinite(mapped).all() and np.min(mapped)>=0
    return mapped

def apply_mapping(source,mapping):
    total=source[:,0,:].sum(axis=1,dtype=np.float64)
    target=mapped_total(total,mapping)
    ratio=np.divide(target,total,out=np.zeros_like(target),where=total>0)
    result=source.copy()
    result[:,0,:]=(source[:,0,:].astype(np.float64)*ratio[:,None]).astype('float32')
    assert np.array_equal(result[:,1,:],source[:,1,:]), 'GG must remain unchanged.'
    if not np.isfinite(result).all() or np.min(result)<0:
        raise ValueError('Bad mapped sample; stop before writing.')
    return result


## Checkpoint 1 — fit and audit three training-only mappings

Each JSON file records the mapping, prepared-data fingerprint, and source training-sample fingerprint. Reruns refuse a changed input rather than silently reusing stale output.


In [ ]:
MAPPINGS={}
for seed in SEEDS:
    path=REPORTS/f'{CAL_ARM}_seed{seed}_mapping.json'
    expected_mapping=fit_mapping(seed)
    if path.exists():
        mapping=json.loads(path.read_text())
        if mapping!=expected_mapping:
            raise ValueError(f'Existing mapping differs: {path}. Choose a new output arm for a changed experiment.')
    else:
        mapping=expected_mapping
        temp=path.with_suffix('.tmp')
        temp.write_text(json.dumps(mapping,indent=2))
        temp.replace(path)
    MAPPINGS[seed]=mapping
    source,idx=load_source(seed,'train')
    adjusted=apply_mapping(source,mapping)
    real_total=DATA['x'][idx,0,:].sum(axis=1)
    before=wasserstein_distance(real_total,source[:,0,:].sum(axis=1))
    after=wasserstein_distance(real_total,adjusted[:,0,:].sum(axis=1))
    print(f'seed {seed}: training-fit daily-load W1 {before:.4f} -> {after:.4f}; '
          f'zero-source days: {mapping["source_zero_days"]}')


## Checkpoint 2 — apply mapping to matched contexts

Write new compressed samples in the standard `outputs/samples/` layout for `train`, `test`, and `privacy`. All original samples remain available. An adjacent provenance JSON records the source file checksum and the new sample checksum. Re-running skips an output only after checking both against the current source. Checkpoints are written atomically.


In [ ]:
def write_sample(path,x,idx):
    temp=path.with_suffix('.tmp')
    with temp.open('wb') as stream: np.savez_compressed(stream,x=x,idx=idx)
    temp.replace(path)

for seed in SEEDS:
    for context in ('train','test','privacy'):
        source_path=SAMPLES/f'{BASE_ARM}_seed{seed}_{context}.npz'
        output_path=SAMPLES/f'{CAL_ARM}_seed{seed}_{context}.npz'
        provenance_path=REPORTS/f'{CAL_ARM}_seed{seed}_{context}_provenance.json'
        source_sha=sha256_file(source_path)
        expected_info={'method':METHOD,'data_sha256':DATA_SHA256,
                       'source_sha256':source_sha,'mapping_sha256':sha256_file(REPORTS/f'{CAL_ARM}_seed{seed}_mapping.json')}
        if output_path.exists() and provenance_path.exists():
            info=json.loads(provenance_path.read_text())
            with np.load(output_path,allow_pickle=False) as saved:
                assert np.array_equal(saved['idx'],context_indices(context)), (seed,context)
                assert saved['x'].shape==(len(saved['idx']),2,48)
            if any(info.get(key)!=value for key,value in expected_info.items()) or info.get('output_sha256')!=sha256_file(output_path):
                raise ValueError(f'Existing output has changed: {output_path}. Preserve it under a new arm name.')
            print('Verified existing:',output_path.name)
            continue
        if provenance_path.exists():
            raise ValueError(f'Missing sample for existing provenance: {output_path}. Inspect before rerunning.')
        source,idx=load_source(seed,context)
        adjusted=apply_mapping(source,MAPPINGS[seed])
        train_max=DATA['x'][DATA['split']==0,0,:].max()
        if adjusted[:,0,:].max()>10*train_max:
            raise ValueError(f'Extreme GC+CL after calibration: seed={seed}, context={context}')
        if output_path.exists():
            # A prior run may have stopped after its atomic archive write.
            with np.load(output_path,allow_pickle=False) as saved:
                if not np.array_equal(saved['idx'],idx) or not np.array_equal(saved['x'],adjusted):
                    raise ValueError(f'Incomplete sample differs from expected output: {output_path}')
        else:
            write_sample(output_path,adjusted,idx)
        info={**expected_info,'output_sha256':sha256_file(output_path),
              'zero_source_days':int((source[:,0,:].sum(axis=1)==0).sum()),
              'max_load_interval_kwh':float(adjusted[:,0,:].max())}
        temp=provenance_path.with_suffix('.tmp')
        temp.write_text(json.dumps(info,indent=2))
        temp.replace(provenance_path)
        print('Saved',output_path.name)


## Checkpoint 3 — fixed evaluation and paired comparison

Use the exact evaluator from the VAE and diffusion experiments. The only changed input is the new arm. Summary metrics average the **same three seeds**; they do not give three independent household splits. The `real_to_real_MAE` is the reference predictor; an observed synthetic MAE close to it is not evidence of superiority.


In [ ]:
for seed in SEEDS:
    for context in ('train','test','privacy'):
        assert (SAMPLES/f'{CAL_ARM}_seed{seed}_{context}.npz').is_file()
    existing_evaluation.evaluate(DATA,SAMPLES,REPORTS,CAL_ARM,seed)

rows=[]
for arm in ('vae_post',BASE_ARM,CAL_ARM):
    for seed in SEEDS:
        path=REPORTS/f'{arm}_seed{seed}.json'
        if not path.exists():
            print('Prior comparison report missing:',path.name)
            continue
        report=json.loads(path.read_text())
        f,u,p=report['fidelity_and_physics'],report['utility'],report['disclosure_probe']
        rows.append({'arm':arm,'seed':seed,
                     'load_daily_W1':f['total_consumption_daily_kwh_wasserstein'],
                     'solar_daily_W1':f['gross_generation_daily_kwh_wasserstein'],
                     'load_peak_W1':f['total_consumption_peak_interval_kwh_wasserstein'],
                     'solar_peak_W1':f['gross_generation_peak_interval_kwh_wasserstein'],
                     'load_peak_q95_gap':f['total_consumption_peak_interval_kwh_synthetic_q95']-f['total_consumption_peak_interval_kwh_real_q95'],
                     'GC_GG_corr_gap':f['gccl_gg_daily_total_correlation_synthetic']-f['gccl_gg_daily_total_correlation_real'],
                     'night_GG_kWh':f['night_solar_kwh'],
                     'synthetic_to_real_MAE':u['synthetic_train_test_real_mae'],
                     'real_to_real_MAE':u['real_train_test_real_mae'],
                     'attack_AUC':p['household_membership_auc']})
results=pd.DataFrame(rows)
display(results.round(4))
display(results.groupby('arm').mean(numeric_only=True).drop(columns='seed').round(4))
if set(SEEDS).issubset(results.loc[results.arm==BASE_ARM,'seed']) and set(SEEDS).issubset(results.loc[results.arm==CAL_ARM,'seed']):
    base=results.loc[results.arm==BASE_ARM].set_index('seed')
    calibrated=results.loc[results.arm==CAL_ARM].set_index('seed')
    delta=calibrated.select_dtypes('number')-base.select_dtypes('number')
    print('Paired change (calibrated minus unchanged diffusion; negative W1/MAE is better):')
    display(delta.round(4))
    assert np.all(np.abs(delta['solar_daily_W1'])<1e-6)
    assert np.all(np.abs(delta['night_GG_kWh'])<1e-6)


## Conditional and tail audit

Global total calibration could obscure errors by season or system size. These fixed diagnostics compare daily-load W1 by calendar month and by three training-defined capacity groups, and inspect peak tails. They are descriptive checks on an already-viewed test split, not extra selection targets.


In [ ]:
capacity_edges=np.quantile(DATA['capacity'][DATA['split']==0],[1/3,2/3])
audit=[]
for seed in SEEDS:
    baseline,idx=load_source(seed,'test')
    with np.load(SAMPLES/f'{CAL_ARM}_seed{seed}_test.npz',allow_pickle=False) as z:
        calibrated,cal_idx=z['x'],z['idx']
    assert np.array_equal(idx,cal_idx)
    real=DATA['x'][idx,0].sum(axis=1)
    before=baseline[:,0].sum(axis=1)
    after=calibrated[:,0].sum(axis=1)
    months=pd.DatetimeIndex(DATA['date'][idx]).month.to_numpy()
    capacity_group=np.searchsorted(capacity_edges,DATA['capacity'][idx],side='right')
    for kind,groups,levels in (('month',months,range(1,13)),('capacity_tertile',capacity_group,range(3))):
        for level in levels:
            mask=groups==level
            if mask.sum()<25: continue
            audit.append({'seed':seed,'group':kind,'value':level,'days':int(mask.sum()),
                          'W1_before':wasserstein_distance(real[mask],before[mask]),
                          'W1_after':wasserstein_distance(real[mask],after[mask])})
audit=pd.DataFrame(audit)
display(audit.groupby(['group','value'])[['days','W1_before','W1_after']].mean().round(3))

baseline,idx=load_source(1,'test')
with np.load(SAMPLES/f'{CAL_ARM}_seed1_test.npz',allow_pickle=False) as z:
    calibrated=z['x']
real=DATA['x'][idx]
fig,axes=plt.subplots(1,2,figsize=(11,3.5))
for label,array in (('held-out real',real),('diffusion v2 + mask',baseline),('load-total calibrated',calibrated)):
    totals=np.sort(array[:,0,:].sum(axis=1))
    peaks=np.sort(array[:,0,:].max(axis=1))
    axes[0].plot(totals,np.linspace(0,1,len(totals)),label=label)
    axes[1].plot(np.quantile(peaks,[.5,.9,.95,.99,.999]),[.5,.9,.95,.99,.999],marker='o',label=label)
axes[0].set(xlabel='Daily GC+CL (kWh)',ylabel='CDF',title='Daily load totals')
axes[1].set(xlabel='Maximum half-hour GC+CL (kWh)',ylabel='Quantile',title='Peak tail')
for axis in axes: axis.legend(fontsize=8)
plt.tight_layout(); plt.show()
print('Seed 1 load peak maxima real / baseline / calibrated:',
      real[:,0,:].max(),baseline[:,0,:].max(),calibrated[:,0,:].max())


## Decision record

Report daily-load W1 **and** peak W1, 95th/99th-percentile peaks, solar fidelity, GC+CL–GG correlation, utility, and the bounded household attack for all three paired seeds. A favorable result is a consistent reduction in daily-load W1 without a material regression in peak fidelity or synthetic-to-real MAE. Training-fit W1 should never substitute for the held-out assessment. If calibrated results degrade peaks or conditional groups, retain the unchanged diffusion arm and describe the tradeoff. Since the test split informed this experiment, confirm any selected method with a future untouched household split before making a final claim.

Compute: existing profiles only; **0 GPU hours** and approximately nine additional compressed sample archives plus small JSON reports. The existing evaluator, especially its PCA-based attack, accounts for most CPU time. No external API calls or fees.
